In [3]:
import os, re, sqlite3
import pandas as pd

DATA_DIR = '/content/data'   # adjust if your folder lives elsewhere

mdb = sqlite3.connect(':memory:')

def mq(sql):
    '''Run a query against the Movies DB and return a DataFrame.'''
    return pd.read_sql_query(sql, mdb)

# Postgres -> SQLite tweaks applied as we read each .sql file
for f in sorted(os.listdir(DATA_DIR)):
    if not f.endswith('.sql'):
        continue
    with open(os.path.join(DATA_DIR, f)) as fh:
        sql = fh.read()
    sql = re.sub(r'DROP SCHEMA.*?CASCADE;', '', sql, flags=re.IGNORECASE | re.DOTALL)
    sql = re.sub(r'CREATE SCHEMA\s+\w+\s*;', '', sql, flags=re.IGNORECASE)
    sql = sql.replace('movies.', '')
    sql = re.sub(r'INT NOT NULL GENERATED BY DEFAULT AS IDENTITY', 'INTEGER', sql)
    sql = re.sub(r'varchar\(\d+\)', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'decimal\(\d+,\d+\)', 'REAL', sql, flags=re.IGNORECASE)
    sql = re.sub(r'BIGINT', 'INTEGER', sql)
    sql = re.sub(r'date\s+DEFAULT', 'TEXT DEFAULT', sql)
    try:
        mdb.executescript(sql)
    except Exception:
        # Some files emit harmless 'no transaction' warnings after auto-commit; ignore.
        pass

# Sanity check
print('Movies DB loaded. Row counts:')
for t in ['movie', 'genre', 'person', 'production_company', 'keyword',
          'movie_genres', 'movie_cast', 'movie_crew', 'movie_company', 'movie_keywords']:
    try:
        n = mdb.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'  {t:22s}: {n:>8,} rows')
    except Exception:
        print(f'  {t:22s}: MISSING')

Movies DB loaded. Row counts:
  movie                 :    4,803 rows
  genre                 :       20 rows
  person                :  104,842 rows
  production_company    :    5,047 rows
  keyword               :    9,794 rows
  movie_genres          :   12,160 rows
  movie_cast            :  106,257 rows
  movie_crew            :  129,581 rows
  movie_company         :   13,677 rows
  movie_keywords        :   36,162 rows


🌟 Task 1: Calculate the Average Budget Growth Rate for Each Production Company


In [4]:
sql_task1 = """
WITH BudgetChanges AS (
    SELECT
        pc.company_name,
        m.budget,
        -- Récupérer le budget du film précédent de la société
        LAG(m.budget) OVER (PARTITION BY pc.company_id ORDER BY m.release_date) as prev_budget
    FROM production_company pc
    JOIN movie_company mc ON pc.company_id = mc.company_id
    JOIN movie m ON mc.movie_id = m.movie_id
    WHERE m.budget > 0
),
GrowthRates AS (
    SELECT
        company_name,
        -- Formule : (Actuel - Précédent) / Précédent
        CAST(budget - prev_budget AS REAL) / prev_budget as growth_rate
    FROM BudgetChanges
    WHERE prev_budget IS NOT NULL AND prev_budget > 0
)
SELECT company_name, AVG(growth_rate) as avg_growth_rate
FROM GrowthRates
GROUP BY company_name
ORDER BY avg_growth_rate DESC
LIMIT 10;
"""
mq(sql_task1)

,company_name,avg_growth_rate
0,Artists Production Group (APG),1.428571e+06
1,Focus Films,5.357135e+05
2,Homegrown Pictures,4.666657e+05
3,Beijing New Picture Film Co. Ltd.,4.272717e+05
4,Muse Productions,2.500003e+05
5,Paramount Animation,1.041666e+05
6,Canadian Film or Video Production Tax Credit (...,7.142869e+04
7,United Artists,5.454650e+04
8,Dimension Films,5.163087e+04
9,Alliance Atlantis Communications,4.762388e+04


🌟 Task 2: Determine the Most Consistently High-Rated Actor


In [5]:
sql_task2 = """
WITH GlobalAvg AS (
    SELECT AVG(vote_average) as avg_rating FROM movie
),
HighRatedMovies AS (
    SELECT m.movie_id
    FROM movie m, GlobalAvg
    WHERE m.vote_average > GlobalAvg.avg_rating
)
SELECT
    p.person_name,
    COUNT(*) as high_rated_movie_count
FROM person p
JOIN movie_cast mc ON p.person_id = mc.person_id
JOIN HighRatedMovies hrm ON mc.movie_id = hrm.movie_id
GROUP BY p.person_id, p.person_name
ORDER BY high_rated_movie_count DESC
LIMIT 10;
"""
mq(sql_task2)

,person_name,high_rated_movie_count
0,Samuel L. Jackson,45
1,Matt Damon,37
2,Robert De Niro,36
3,Morgan Freeman,35
4,Bruce Willis,31
5,Tom Hanks,30
6,Brad Pitt,30
7,Denzel Washington,29
8,Willem Dafoe,29
9,Johnny Depp,28


🌟 Task 3: Calculate the Rolling Average Revenue for Each Genre


In [6]:
sql_task3 = """
SELECT
    g.genre_name,
    m.title,
    m.release_date,
    m.revenue,
    -- Moyenne sur le film actuel et les 2 précédents
    AVG(m.revenue) OVER (
        PARTITION BY g.genre_name
        ORDER BY m.release_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) as rolling_avg_revenue
FROM genre g
JOIN movie_genres mg ON g.genre_id = mg.genre_id
JOIN movie m ON mg.movie_id = m.movie_id
WHERE m.revenue > 0
ORDER BY g.genre_name, m.release_date
LIMIT 20;
"""
mq(sql_task3)

,genre_name,title,release_date,revenue,rolling_avg_revenue
0,Action,Hell's Angels,1930-11-15,8000000,8.000000e+06
1,Action,The Charge of the Light Brigade,1936-10-20,2736000,5.368000e+06
2,Action,Sands of Iwo Jima,1949-12-14,7800000,6.178667e+06
3,Action,Annie Get Your Gun,1950-05-17,8000000,6.178667e+06
4,Action,The Greatest Show on Earth,1952-01-10,36000000,1.726667e+07
5,Action,七人の侍,1954-04-26,271841,1.475728e+07
6,Action,Love Me Tender,1956-11-15,9000000,1.509061e+07
7,Action,The Misfits,1961-01-01,8200000,5.823947e+06
8,Action,The Longest Day,1962-09-25,50100000,2.243333e+07
9,Action,Dr. No,1962-10-04,59600000,3.930000e+07


🌟 Task 4: Identify the Highest-Grossing Movie Series


In [7]:
sql_task4 = """
WITH SeriesRevenue AS (
    SELECT
        k.keyword_name,
        SUM(m.revenue) as total_series_revenue,
        COUNT(m.movie_id) as movie_count
    FROM keyword k
    JOIN movie_keywords mk ON k.keyword_id = mk.keyword_id
    JOIN movie m ON mk.movie_id = m.movie_id
    GROUP BY k.keyword_id, k.keyword_name
    HAVING COUNT(m.movie_id) > 1 -- Une série doit avoir au moins 2 films
)
SELECT keyword_name as series_name, total_series_revenue
FROM SeriesRevenue
ORDER BY total_series_revenue DESC
LIMIT 10;
"""
mq(sql_task4)

,series_name,total_series_revenue
0,duringcreditsstinger,57827617707
1,3d,38821575006
2,aftercreditsstinger,37735182966
3,based on novel,28330574606
4,sequel,26416350086
5,superhero,26398102240
6,dystopia,22093114379
7,based on comic book,21464756515
8,marvel comic,19447081252
9,woman director,16140425506
